# HiGANCon — Kaggle training
Notebook chuẩn cho luồng GitHub → Hugging Face HDF5 → train → eval. Bật GPU và Internet trong Kaggle trước khi Run All.

In [ ]:
# Chỉ sửa cell này cho mỗi lần chạy.
REPO_URL = "https://github.com/lenhanbo/HiGANCon.git"
GIT_BRANCH = "main"
REPO_DIR = "/kaggle/working/HiGANCon"

HF_REPO = "lenhanbo/higancon"
HDF5_PREFIX = ""  # Ví dụ: "data" nếu ba file HDF5 nằm trong data/ trên Hub.
HDF5_ROOT = "/kaggle/working/higan_hdf5"
MANIFEST_ROOT = "/kaggle/working/data_higan_hdf5"

RUN_MODE = "supcon"  # baseline | supcon | contrastive | ctc | full
SUPCON_TEXT_MODE = "per_item"  # per_item | shared
RESUME_PATH = ""  # Chỉ dùng last.pt tạo bởi config.yaml và cùng mode/text mode.

MAX_HOURS = 10
MAX_STEPS = 30000  # 0 = không giới hạn global step.
EVAL_MAX_SAMPLES = 0  # 0 = toàn bộ test set.
FID_KID_PROTOCOL = "higan"
EVAL_SEED = 123456
SKIP_HWD = True
HWD_CHECKPOINT = ""
RECOGNIZER_CHECKPOINT = ""  # Cần checkpoint độc lập để CER/WER có giá trị báo cáo.

In [ ]:
# Clone lần đầu; các lần sau cập nhật đúng branch bằng fast-forward.
!if [ -d "{REPO_DIR}/.git" ]; then git -C "{REPO_DIR}" checkout "{GIT_BRANCH}" && git -C "{REPO_DIR}" pull --ff-only origin "{GIT_BRANCH}"; else git clone --branch "{GIT_BRANCH}" "{REPO_URL}" "{REPO_DIR}"; fi
%cd {REPO_DIR}
!git rev-parse HEAD
!python -m pip install -q -r requirements-kaggle.txt

## Tải và chuẩn bị HDF5

In [ ]:
!python scripts/download_hf_hdf5.py "{HF_REPO}" --repo-type dataset --prefix "{HDF5_PREFIX}" --output-dir "{HDF5_ROOT}"
!bash scripts/kaggle_setup.sh "{HDF5_ROOT}" higan_hdf5 "{MANIFEST_ROOT}"

## Smoke test
Chạy một batch với đúng loss mode và SupCon text mode trước khi train dài.

In [ ]:
import shlex
import subprocess
from pathlib import Path
from IPython.display import Image, display

RUN_MODE = RUN_MODE.strip().lower()
SUPCON_TEXT_MODE = SUPCON_TEXT_MODE.strip().lower()
assert RUN_MODE in {"baseline", "supcon", "contrastive", "ctc", "full"}
assert SUPCON_TEXT_MODE in {"shared", "per_item"}
loss_args = {
    "baseline": ["--no", "supcon", "--no", "ctc-guided"],
    "supcon": ["--no", "ctc-guided"],
    "contrastive": ["--no", "ctc-guided"],
    "ctc": ["--no", "supcon"],
    "full": [],
}[RUN_MODE]
text_flag = "--supcon-shared-text" if SUPCON_TEXT_MODE == "shared" else "--supcon-per-item-text"
smoke_name = f"{RUN_MODE}-{SUPCON_TEXT_MODE}" if RUN_MODE in {"supcon", "contrastive", "full"} else RUN_MODE
SMOKE_OUTPUT = f"/kaggle/working/smoke/{smoke_name}"
smoke_args = ["python", "train.py", "--config", "configs/config.yaml", "--dataset-root", HDF5_ROOT, "--dataset-format", "higan_hdf5", "--manifest-root", MANIFEST_ROOT, "--output", SMOKE_OUTPUT, "--smoke-test", text_flag] + loss_args
print(shlex.join(smoke_args))
subprocess.run(smoke_args, check=True)
preview = Path(SMOKE_OUTPUT) / "previews" / "step_00000001.png"
if preview.exists():
    display(Image(filename=str(preview)))

## Train và tự động eval
Script lưu checkpoint có thể resume, sau đó eval đúng một lần và ghi `evaluation.json`.

In [ ]:
run_args = ["bash", "scripts/kaggle_run.sh", "--dataset-root", HDF5_ROOT, "--dataset-format", "higan_hdf5", "--manifest-root", MANIFEST_ROOT, "--run-mode", RUN_MODE, text_flag, "--max-hours", str(MAX_HOURS), "--max-steps", str(MAX_STEPS), "--eval-max-samples", str(EVAL_MAX_SAMPLES), "--fid-kid-protocol", FID_KID_PROTOCOL, "--eval-seed", str(EVAL_SEED)]
if RESUME_PATH:
    run_args += ["--resume", RESUME_PATH]
if RECOGNIZER_CHECKPOINT:
    run_args += ["--recognizer-checkpoint", RECOGNIZER_CHECKPOINT]
if HWD_CHECKPOINT:
    run_args += ["--hwd-checkpoint", HWD_CHECKPOINT]
if SKIP_HWD:
    run_args += ["--skip-hwd"]
print(shlex.join(run_args))
subprocess.run(run_args, check=True)

## Kết quả

In [ ]:
output_name = f"{RUN_MODE}-{SUPCON_TEXT_MODE}" if RUN_MODE in {"supcon", "contrastive", "full"} else RUN_MODE
OUTPUT_DIR = Path("/kaggle/working/outputs") / output_name
preview_files = sorted((OUTPUT_DIR / "previews").glob("*.png"))
if preview_files:
    display(Image(filename=str(preview_files[-1])))
evaluation_path = OUTPUT_DIR / "evaluation.json"
if evaluation_path.exists():
    print(evaluation_path.read_text())
print(OUTPUT_DIR)